In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/veeraj16/18k-counterfactual-dataset/final_dataset_18k.csv
/kaggle/input/datasets/veeraj16/6k-samples-dataset/hate_speech_dataset_6k.csv


In [1]:
# %% [markdown]
# # Identity-Counterfactual Data Augmentation — Kaggle T4×2
# **Qwen2.5-3B-Instruct + vLLM tensor parallelism across both GPUs**
#
# **IMPORTANT**: Run Cell 1 alone first, then use *Restart & Run All* from Cell 2 onward.

# %% [markdown]
# ## Cell 1 — Fix environment (run this cell alone, then restart kernel)

# %%
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args], stdout=subprocess.DEVNULL)

print("Step 1/4: Pinning numpy<2.0 ...")
pip("install", "-q", "--upgrade", "numpy==1.26.4")

print("Step 2/4: Installing polars ...")
pip("install", "-q", "polars>=0.20")

print("Step 3/4: Installing transformers ...")
pip("install", "-q", "--upgrade", "transformers==4.46.3", "huggingface_hub>=0.24")

print("Step 4/4: Installing vLLM ...")
pip("install", "-q", "vllm==0.6.3.post1",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121")

print("\n✅ All packages installed.")
print("⚠️  NOW: Kernel → Restart, then run all remaining cells.")

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

Step 1/4: Pinning numpy<2.0 ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<

Step 2/4: Installing polars ...
Step 3/4: Installing transformers ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


Step 4/4: Installing vLLM ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.4.0+cu121 which is incompatible.



✅ All packages installed.
⚠️  NOW: Kernel → Restart, then run all remaining cells.


{'status': 'ok', 'restart': True}

In [1]:
# %% [markdown]
# ---
# ## Cell 2 — Verify environment

# %%
import numpy as np
import torch

assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2, 0), \
    f"numpy {np.__version__} detected — must be <2.0. Re-run Cell 1."

print(f"✅ numpy  {np.__version__}")
print(f"✅ torch  {torch.__version__}  |  CUDA {torch.version.cuda}")

n_gpus = torch.cuda.device_count()
print(f"✅ GPUs   {n_gpus}")
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"   GPU {i}: {p.name}  ({p.total_memory / 1e9:.1f} GB VRAM)")

assert n_gpus >= 2, "Need T4×2. Go to Settings → Accelerator → GPU T4×2."

# %% [markdown]
# ## Cell 3 — Imports & configuration

# %%
import os, re, json, hashlib, time, logging
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import polars as pl

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Paths ───────────────────────────────────────────────────────────────────
INPUT_CSV  = Path("/kaggle/input/datasets/veeraj16/6k-samples-dataset/hate_speech_dataset_6k.csv")
OUTPUT_CSV = Path("/kaggle/working/final_dataset_18k.csv")

# ── Model config ────────────────────────────────────────────────────────────
MODEL_ID            = "Qwen/Qwen2.5-3B-Instruct"

TENSOR_PARALLEL     = 2
MAX_MODEL_LEN       = 1024
MAX_NEW_TOKENS      = 128
TEMPERATURE         = 0.25
TOP_P               = 0.9
GPU_MEM_UTIL        = 0.92

log.info("Config loaded. Model: %s | GPUs: %d", MODEL_ID, TENSOR_PARALLEL)

# %% [markdown]
# ## Cell 4 — Identity dictionaries & pre-compiled regex

# %%
IDENTITY_AXES: dict[str, list[str]] = {
    "race_ethnicity": [
        "Black", "White", "Asian", "Latino", "Hispanic", "Arab",
        "African", "European", "Indian", "Chinese", "Japanese", "Korean",
        "Mexican", "Jewish", "Native American", "Indigenous", "Pacific Islander",
        "Caribbean", "Middle Eastern", "Southeast Asian", "South Asian",
        "East Asian", "Caucasian", "biracial", "multiracial",
    ],
    "religion": [
        "Muslim", "Christian", "Jewish", "Hindu", "Buddhist", "Sikh",
        "Atheist", "Catholic", "Protestant", "Mormon", "Evangelical",
        "Orthodox", "Sunni", "Shia", "Jew", "Islam", "Islamic",
    ],
    "gender_sexuality": [
        "women", "men", "transgender", "nonbinary", "gay", "lesbian",
        "bisexual", "queer", "straight", "heterosexual", "homosexual",
        "female", "male", "trans", "LGBTQ", "cisgender", "asexual",
    ],
    "nationality": [
        "American", "Mexican", "Chinese", "Indian", "Nigerian", "Brazilian",
        "British", "French", "German", "Russian", "Iranian", "Iraqi",
        "Syrian", "Afghan", "Pakistani", "Somali", "Ethiopian",
        "Japanese", "Korean", "Colombian", "Venezuelan", "Cuban",
    ],
    "disability": [
        "disabled", "blind", "deaf", "autistic", "mentally ill",
        "wheelchair user", "handicapped", "special needs",
    ],
    "age": [
        "old", "young", "elderly", "teenager", "millennial",
        "boomer", "Gen Z", "senior", "youth",
    ],
}

SLUR_TO_IDENTITY: dict[str, str] = {
    "nigger": "Black",   "nigga": "Black",    "niggas": "Black",   "negro": "Black",
    "darkie": "Black",   "coon": "Black",     "spook": "Black",    "negroes": "Black",
    "chink": "Asian",    "gook": "Asian",     "slant": "Asian",    "zipperhead": "Asian",
    "jap": "Japanese",   "nip": "Japanese",
    "spic": "Latino",    "spick": "Latino",   "beaner": "Latino",  "wetback": "Latino",
    "gringo": "White",   "cracker": "White",  "redneck": "White",  "honky": "White",
    "kike": "Jewish",    "yid": "Jewish",     "hebe": "Jewish",
    "muzzie": "Muslim",  "towelhead": "Muslim","raghead": "Muslim", "muzzy": "Muslim",
    "fag": "gay",        "faggot": "gay",     "dyke": "lesbian",   "tranny": "transgender",
    "homo": "homosexual",
    "whore": "women",    "bitch": "women",    "slut": "women",     "cunt": "women",
    "retard": "disabled","retarded": "disabled","cripple": "disabled",
}

TARGET_GROUP_TO_AXIS: dict[str, Optional[str]] = {
    "race/ethnicity":              "race_ethnicity",
    "religion":                    "religion",
    "gender":                      "gender_sexuality",
    "sexual_orientation":          "gender_sexuality",
    "national_origin/citizenship": "nationality",
    "disability":                  "disability",
    "age":                         "age",
    "multiple/none":               None,
}

TERM_TO_AXIS: dict[str, str] = {}
for _ax, _terms in IDENTITY_AXES.items():
    for _t in _terms:
        TERM_TO_AXIS[_t.lower()] = _ax
for _slur, _id in SLUR_TO_IDENTITY.items():
    if (_ax := TERM_TO_AXIS.get(_id.lower())):
        TERM_TO_AXIS[_slur.lower()] = _ax


@dataclass(slots=True)
class DetectedTerm:
    term: str; axis: str; identity: str
    start: int; end: int; is_slur: bool

@dataclass(slots=True)
class Replacement:
    original_term: str; replacement: str
    axis: str; original_identity: str


def _compile_patterns():
    slur_pats, id_pats = [], []
    for slur in sorted(SLUR_TO_IDENTITY, key=len, reverse=True):
        identity = SLUR_TO_IDENTITY[slur]
        axis     = TERM_TO_AXIS.get(identity.lower(), "unknown")
        slur_pats.append((
            re.compile(r'\b' + re.escape(slur) + r'(?:s|es|ed|ing)?\b', re.I),
            identity, axis,
        ))
    for ax, terms in IDENTITY_AXES.items():
        for term in sorted(terms, key=len, reverse=True):
            id_pats.append((
                re.compile(r'\b' + re.escape(term) + r'(?:s|es)?\b', re.I),
                term, ax,
            ))
    return slur_pats, id_pats

_SLUR_PATS, _ID_PATS = _compile_patterns()

_CLEAN_PREFIX_RE = re.compile(
    r'^(?:rewritten[\s\w]*?|output|result|here[\s\w]*?|text|sentence|answer|cf\s*\d*|'
    r'counterfactual[\s\w]*?):\s*',
    re.I,
)
_THINK_BLOCK_RE = re.compile(r'<think>.*?</think>', re.S)

log.info("✅ Identity dictionaries & regex patterns ready (%d slur + %d identity patterns)",
         len(_SLUR_PATS), len(_ID_PATS))

# %% [markdown]
# ## Cell 5 — Regex helpers (detection only — used to build LLM prompts)
# NOTE: regex_swap / try_regex_cf are removed. Detection is kept because
# build_explicit_prompt needs to know which terms exist and what to swap them to.

# %%
def detect_identity_terms(text: str) -> list[DetectedTerm]:
    found: list[DetectedTerm] = []
    occupied: set[int] = set()
    lower = text.lower()

    def _claim(s: int, e: int) -> bool:
        span = set(range(s, e))
        if span & occupied: return False
        occupied.update(span); return True

    for pat, identity, axis in _SLUR_PATS:
        for m in pat.finditer(lower):
            if _claim(m.start(), m.end()):
                found.append(DetectedTerm(
                    text[m.start():m.end()], axis, identity,
                    m.start(), m.end(), True,
                ))

    for pat, term, axis in _ID_PATS:
        for m in pat.finditer(lower):
            if _claim(m.start(), m.end()):
                found.append(DetectedTerm(
                    text[m.start():m.end()], axis, term,
                    m.start(), m.end(), False,
                ))

    found.sort(key=lambda d: d.start)
    return found


def _pick_replacement(identity: str, axis: str, cf_index: int, seed: str) -> Optional[str]:
    candidates = [t for t in IDENTITY_AXES.get(axis, []) if t.lower() != identity.lower()]
    if not candidates:
        return None
    h0 = int(hashlib.md5(f"{seed}_0".encode()).hexdigest(), 16) % len(candidates)
    if cf_index == 0:
        return candidates[h0]
    h1 = int(hashlib.md5(f"{seed}_1".encode()).hexdigest(), 16) % len(candidates)
    if len(candidates) > 1 and h1 == h0:
        h1 = (h1 + 1) % len(candidates)
    return candidates[h1]


def _pick_implicit_identity(sample_id: str, target_group: str, cf_index: int) -> tuple[str, str]:
    axis = TARGET_GROUP_TO_AXIS.get(target_group)
    if axis and axis in IDENTITY_AXES:
        terms = IDENTITY_AXES[axis]
        h0 = int(hashlib.md5(f"{sample_id}_0_impl".encode()).hexdigest(), 16) % len(terms)
        if cf_index == 0: return terms[h0], axis
        h1 = int(hashlib.md5(f"{sample_id}_1_impl".encode()).hexdigest(), 16) % len(terms)
        if len(terms) > 1 and h1 == h0: h1 = (h1 + 1) % len(terms)
        return terms[h1], axis
    axes = list(IDENTITY_AXES.keys())
    h = int(hashlib.md5(f"{sample_id}_{cf_index}_noax".encode()).hexdigest(), 16)
    ax = axes[h % len(axes)]
    if cf_index == 1:
        h0 = int(hashlib.md5(f"{sample_id}_0_noax".encode()).hexdigest(), 16)
        if ax == axes[h0 % len(axes)] and len(axes) > 1:
            ax = axes[(h + 1) % len(axes)]
    terms = IDENTITY_AXES[ax]
    return terms[h % len(terms)], ax


def _build_row(sample_id: str, cf_text: str, row: dict, cf_index: int) -> dict:
    return {
        "original_sample_id": sample_id,
        "counterfactual_id":  f"{sample_id}_cf{cf_index + 1}",
        "text":               cf_text,
        "class_label":        row["class_label"],
        "target_group":       row.get("target_group", "multiple/none"),
        "polarity":           row.get("polarity", "non-hate"),
        "hate_score":         None,
        "confidence":         None,
        "cf_type":            f"counterfactual_{cf_index + 1}",
        "t2i_prompt":         "",
    }


def _injection_fallback(text: str, target_identity: str, cf_index: int) -> str:
    words = text.split()
    if len(words) >= 6:
        mid = len(words) // 3
        words.insert(mid, f"({target_identity})")
        return " ".join(words)
    if cf_index == 0:
        return f"{text} [about {target_identity} people]"
    return f"[{target_identity}]: {text}"


log.info("✅ Detection helpers ready")

# %% [markdown]
# ## Cell 6 — Build full LLM queue (all 12k counterfactuals go to the model)
# Phase 1 CPU regex swap is removed. Every row × 2 CFs is sent to the LLM.

# %%
df_source = pl.read_csv(str(INPUT_CSV))
log.info("Source: %d rows | classes: %s",
         len(df_source),
         df_source["class_label"].value_counts().to_dicts())

rows = df_source.to_dicts()

llm_queue: list[dict] = []

for row in rows:
    detected = detect_identity_terms(row["text"])
    for cf_idx in range(2):
        # If explicit identity terms found, tell the LLM exactly what to swap.
        # Otherwise fall back to implicit: ask the LLM to rewrite for a chosen group.
        if detected:
            reps = [
                Replacement(d.term, rep, d.axis, d.identity)
                for d in detected
                if (rep := _pick_replacement(d.identity, d.axis, cf_idx, d.term))
            ]
        else:
            reps = []

        target_identity, axis = _pick_implicit_identity(
            row["sample_id"], row.get("target_group", "multiple/none"), cf_idx
        )

        llm_queue.append({
            "row":             row,
            "cf_index":        cf_idx,
            "target_identity": target_identity,
            "axis":            axis,
            "replacements":    reps,   # non-empty → explicit prompt; empty → implicit prompt
        })

log.info("LLM queue built: %d prompts (%d rows × 2 CFs)", len(llm_queue), len(rows))

# %% [markdown]
# ## Cell 7 — Load vLLM on both T4s

# %%
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

log.info("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

log.info("Loading vLLM engine  (tensor_parallel_size=%d) …", TENSOR_PARALLEL)
t0 = time.time()

llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype="float16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    trust_remote_code=True,
    enforce_eager=False,
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    stop=["<|im_end|>", "<|endoftext|>", "\n\n\n"],
    skip_special_tokens=True,
    repetition_penalty=1.1,
)

log.info("✅ vLLM engine ready in %.1f s", time.time() - t0)

# %% [markdown]
# ## Cell 8 — Prompt engineering

# %%
_SYS_EXPLICIT = (
    "You are a dataset augmentation tool for hate-speech research.\n"
    "You swap identity group names in sentences. Follow instructions exactly.\n"
    "Output ONLY the rewritten sentence. No explanation. No quotes. No prefix."
)

_SYS_IMPLICIT = (
    "You are a dataset augmentation tool for hate-speech research.\n"
    "You rewrite sentences to reference a different identity group.\n"
    "Preserve original meaning, tone, and structure exactly.\n"
    "Output ONLY the rewritten sentence. No explanation. No quotes. No prefix."
)


def _chat_prompt(system: str, user: str) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def build_explicit_prompt(text: str, replacements: list[Replacement]) -> str:
    swaps = "\n".join(f'• "{r.original_term}" → "{r.replacement}"' for r in replacements)
    user = (
        "Swap these identity terms and output ONLY the result.\n\n"
        "Example:\n"
        "Swap: • \"Black\" → \"Asian\"\n"
        "Sentence: All Black people should leave.\n"
        "Output: All Asian people should leave.\n\n"
        f"Swap:\n{swaps}\n"
        f"Sentence: {text}\n"
        "Output:"
    )
    return _chat_prompt(_SYS_EXPLICIT, user)


def build_implicit_prompt(
    text: str,
    target_identity: str,
    axis: str,
    polarity: str,
) -> str:
    tone = "hateful/hostile" if polarity == "hate" else "neutral or counter-speech"
    axis_label = axis.replace("_", " ")
    user = (
        "Example:\n"
        "Rewrite for 'Muslim' (religion), hateful/hostile tone.\n"
        "Sentence: Those people ruin every neighbourhood.\n"
        "Output: Muslims ruin every neighbourhood.\n\n"
        f"Rewrite the sentence so it refers to '{target_identity}' ({axis_label}). "
        f"Keep the {tone} tone. If no group is mentioned, insert a natural reference. "
        f"Output ONLY the rewritten sentence.\n\n"
        f"Sentence: {text}\n"
        "Output:"
    )
    return _chat_prompt(_SYS_IMPLICIT, user)


def clean_output(raw: str, original: str) -> str:
    text = _THINK_BLOCK_RE.sub("", raw).strip()
    text = text.strip('"').strip("'")
    text = _CLEAN_PREFIX_RE.sub("", text).strip().strip('"').strip("'")

    if "\n\n" in text:
        text = text.split("\n\n")[0].strip()
    text = text.strip()

    if not text or len(text) < 4:
        return ""

    ratio = len(text) / max(len(original), 1)
    if not (0.25 <= ratio <= 3.0):
        return ""

    non_ascii = sum(
        1 for c in text
        if ord(c) > 127
        and not (0x1F300 <= ord(c) <= 0x1FAFF or 0x2600 <= ord(c) <= 0x27BF)
    )
    if non_ascii / max(len(text), 1) > 0.08:
        return ""

    if re.match(r'^example\s*:', text, re.I):
        return ""

    return text


log.info("✅ Prompt templates ready")

# %% [markdown]
# ## Cell 9 — LLM batched inference (all 12k prompts)

# %%
def run_llm(queue: list[dict]) -> list[dict]:
    if not queue:
        log.info("LLM queue is empty — nothing to do.")
        return []

    log.info("Building %d prompts …", len(queue))
    prompts: list[str] = []

    for item in queue:
        reps      = item["replacements"]
        text      = item["row"]["text"]
        target_id = item["target_identity"]
        axis      = item["axis"]
        polarity  = item["row"].get("polarity", "non-hate")

        # Explicit prompt when the LLM has concrete swap targets;
        # implicit prompt when the text has no detectable identity terms.
        if reps:
            prompts.append(build_explicit_prompt(text, reps))
        else:
            prompts.append(build_implicit_prompt(text, target_id, axis, polarity))

    log.info("Running vLLM on %d prompts across %d GPUs …", len(prompts), TENSOR_PARALLEL)
    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0
    log.info("vLLM complete in %.1f s  (%.1f prompts/s)", elapsed, len(prompts) / elapsed)

    results: list[dict] = []
    fallback_count = 0

    for item, output in zip(queue, outputs):
        row       = item["row"]
        cf_idx    = item["cf_index"]
        target_id = item["target_identity"]
        raw_text  = output.outputs[0].text if output.outputs else ""

        cf_text = clean_output(raw_text, row["text"])

        if cf_text and cf_text.strip().lower() == row["text"].strip().lower():
            cf_text = ""

        if not cf_text:
            fallback_count += 1
            cf_text = _injection_fallback(row["text"], target_id, cf_idx)

        results.append(_build_row(row["sample_id"], cf_text, row, cf_idx))

    log.info(
        "LLM inference done — %d generated  |  %d injection fallbacks (%.1f%%)",
        len(results), fallback_count, fallback_count / max(len(results), 1) * 100,
    )
    return results


llm_results = run_llm(llm_queue)

# %% [markdown]
# ## Cell 10 — Assemble, validate, and save

# %%
log.info("Assembling final dataset …")

original_rows = [
    {
        "original_sample_id": r["sample_id"],
        "counterfactual_id":  r["sample_id"],
        "text":               r["text"],
        "class_label":        r["class_label"],
        "target_group":       r["target_group"],
        "polarity":           r["polarity"],
        "hate_score":         r["hate_score"],
        "confidence":         r["confidence"],
        "cf_type":            "original",
        "t2i_prompt":         "",
    }
    for r in rows
]

# llm_results now contains all 12k counterfactuals (no phase1_results)
all_rows = original_rows + llm_results
df_final = pl.DataFrame(all_rows)

_cf_order = {"original": 0, "counterfactual_1": 1, "counterfactual_2": 2}
df_final = (
    df_final
    .with_columns(
        pl.col("cf_type").replace_strict(_cf_order, default=3).alias("_sort")
    )
    .sort(["original_sample_id", "_sort"])
    .drop("_sort")
)

# Integrity: every original must have exactly 3 rows
group_counts = df_final.group_by("original_sample_id").len()
orphans      = group_counts.filter(pl.col("len") != 3)
if len(orphans):
    log.warning("%d IDs without exactly 3 variants — dropping.", len(orphans))
    valid_ids = group_counts.filter(pl.col("len") == 3)["original_sample_id"]
    df_final  = df_final.filter(pl.col("original_sample_id").is_in(valid_ids))

df_final.write_csv(str(OUTPUT_CSV))

log.info("=" * 60)
log.info("SAVED  %d rows → %s", df_final.shape[0], OUTPUT_CSV)
log.info("cf_type:\n%s",       df_final["cf_type"].value_counts())
log.info("class_label:\n%s",   df_final["class_label"].value_counts())
log.info("=" * 60)

# %% [markdown]
# ## Cell 11 — Quality audit

# %%
print("=" * 70)
print("QUALITY AUDIT — 5 random samples per CF type")
print("=" * 70)

orig_lookup = {r["sample_id"]: r["text"] for r in rows}

for cf_type in ["counterfactual_1", "counterfactual_2"]:
    samples = df_final.filter(pl.col("cf_type") == cf_type).sample(5, seed=42).to_dicts()
    print(f"\n── {cf_type.upper()} ──")
    for r in samples:
        orig = orig_lookup.get(r["original_sample_id"], "?")
        print(f"\n  ID    : {r['original_sample_id']}")
        print(f"  CLASS : {r['class_label']}  |  TARGET: {r['target_group']}")
        print(f"  ORIG  : {orig[:110]}")
        print(f"  CF    : {r['text'][:110]}")

cf_only  = df_final.filter(pl.col("cf_type") != "original")
injected = cf_only.filter(
    pl.col("text").str.contains(r'\[about .+ people\]') |
    pl.col("text").str.contains(r'^\[.+\]:') |
    pl.col("text").str.contains(r'\(.+\)')
)
inj_pct = len(injected) / max(len(cf_only), 1) * 100
print(f"\n── Injection fallback rate: {len(injected)} / {len(cf_only)} ({inj_pct:.1f}%)")
if inj_pct > 15:
    print("⚠️  High fallback rate — consider lowering TEMPERATURE or increasing MAX_NEW_TOKENS.")
else:
    print("✅ Fallback rate within acceptable range.")


def _spot_check_identity_presence(df: pl.DataFrame, n: int = 50) -> float:
    sample = df.filter(pl.col("cf_type") != "original").sample(min(n, len(df)), seed=99)
    hits = 0
    for row in sample.to_dicts():
        if detect_identity_terms(row["text"]):
            hits += 1
    return hits / max(len(sample), 1)

id_hit_rate = _spot_check_identity_presence(df_final)
print(f"\n── Identity-term presence in CFs: {id_hit_rate:.1%}")
if id_hit_rate < 0.5:
    print("⚠️  Low identity-term presence — many CFs may be implicit-only. Review prompts.")
else:
    print("✅ Identity-term presence looks healthy.")

print(f"\n✅ Dataset saved → {OUTPUT_CSV}")

✅ numpy  1.26.4
✅ torch  2.4.0+cu121  |  CUDA 12.1
✅ GPUs   2
   GPU 0: Tesla T4  (15.6 GB VRAM)
   GPU 1: Tesla T4  (15.6 GB VRAM)


09:41:56 [INFO] Config loaded. Model: Qwen/Qwen2.5-3B-Instruct | GPUs: 2
09:41:56 [INFO] ✅ Identity dictionaries & regex patterns ready (41 slur + 98 identity patterns)
09:41:56 [INFO] ✅ Detection helpers ready
09:41:56 [INFO] Source: 6000 rows | classes: [{'class_label': 'ambiguous', 'count': 750}, {'class_label': 'hate_other', 'count': 750}, {'class_label': 'offensive_non_hate', 'count': 750}, {'class_label': 'hate_race', 'count': 750}, {'class_label': 'hate_religion', 'count': 750}, {'class_label': 'counter_speech', 'count': 750}, {'class_label': 'neutral_discussion', 'count': 750}, {'class_label': 'hate_gender', 'count': 750}]
09:41:59 [INFO] LLM queue built: 12000 prompts (6000 rows × 2 CFs)
2026-03-07 09:42:04.994349: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772876525.209099     166 cuda_dnn.cc:8579] Unable to register cuDNN fac

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 03-07 09:43:57 model_runner.py:1067] Loading model weights took 2.9347 GB
(VllmWorkerProcess pid=233) INFO 03-07 09:44:01 model_runner.py:1067] Loading model weights took 2.9347 GB
INFO 03-07 09:44:02 distributed_gpu_executor.py:57] # GPU blocks: 31728, # CPU blocks: 14563
INFO 03-07 09:44:02 distributed_gpu_executor.py:61] Maximum concurrency for 1024 tokens per request: 495.75x
INFO 03-07 09:44:06 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 03-07 09:44:06 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
(VllmWorkerProcess pid=233) INFO 03-07 09:44:06 model_runner.py:1395] Capturing the model for

09:44:43 [INFO] ✅ vLLM engine ready in 130.2 s
09:44:43 [INFO] ✅ Prompt templates ready
09:44:43 [INFO] Building 12000 prompts …
09:44:43 [INFO] Running vLLM on 12000 prompts across 2 GPUs …
Processed prompts: 100%|██████████| 12000/12000 [12:06<00:00, 16.51it/s, est. speed input: 2684.56 toks/s, output: 671.03 toks/s]
09:56:58 [INFO] vLLM complete in 735.4 s  (16.3 prompts/s)
09:56:58 [INFO] LLM inference done — 12000 generated  |  229 injection fallbacks (1.9%)
09:56:59 [INFO] Assembling final dataset …
09:56:59 [INFO] ============================================================
09:56:59 [INFO] SAVED  18000 rows → /kaggle/working/final_dataset_18k.csv
09:56:59 [INFO] cf_type:
shape: (3, 2)
┌──────────────────┬───────┐
│ cf_type          ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ counterfactual_1 ┆ 6000  │
│ counterfactual_2 ┆ 6000  │
│ original         ┆ 6000  │
└──────────────────┴───────┘
09:56:59 [INFO] class_label:
shape: (8

QUALITY AUDIT — 5 random samples per CF type

── COUNTERFACTUAL_1 ──

  ID    : HS_COUNTERSPEECH_0205
  CLASS : counter_speech  |  TARGET: religion
  ORIG  : waking up with good vibes! waahh...EidMubarak to all muslim brothers and sisters around the globe 🎉🎉🎉
  CF    : waking up with good vibes! waahh...EidMubarak to all Catholic brothers and sisters around the globe 🎉🎉🎉

  ID    : HS_HATEGENDER_0324
  CLASS : hate_gender  |  TARGET: gender
  ORIG  : I'd slap the piss out of you then proceed to rape you. Then I'd call all my friends over and let them rape you
  CF    : I'd slap the piss out of you then proceed to rape you. Then I'd call all my lesbian friends over and let them 

  ID    : HS_HATERACE_0234
  CLASS : hate_race  |  TARGET: race/ethnicity
  ORIG  : Now we just need Florida to let us do this with all these piece of shit niggers am I right
  CF    : Now we just need Florida to let us do this with all these piece of shit East Asians am I right

  ID    : HS_COUNTERSPEECH_0156

## Image-Gen

In [1]:
#latest
# %% [markdown]
# ---
# ## Hybrid T2I Prompt Generation Pipeline  
# Combines high-quality prompts from local code with vLLM GPU speed from Kaggle notebook.
# Model: Qwen/Qwen2.5-3B-Instruct (Kaggle model only)
#
# ---

# %% [markdown]
# ## Cell 1 — Imports & Logging

# %%
import os
import re
import json
import time
import random
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple

import polars as pl
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
log.info("✅ Imports ready")


# %% [markdown]
# ## Cell 2 — Global Config

# %%

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_PATH    = "/kaggle/input/datasets/veeraj16/18k-counterfactual-dataset/final_dataset_18k.csv"
OUTPUT_CSV      = "/kaggle/working/final_dataset_18k_t2i_hybrid.csv"
CHECKPOINT_FILE = "/kaggle/working/t2i_checkpoint.json"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen2.5-3B-Instruct"
TENSOR_PARALLEL = 2          # Uses both GPUs via vLLM tensor parallelism
MAX_MODEL_LEN   = 2048
GPU_MEM_UTIL    = 0.92

# ── T2I Generation ────────────────────────────────────────────────────────────
T2I_MAX_NEW_TOKENS       = 200    # Generous budget for rich prompts
T2I_TEMPERATURE          = 0.30   # Low temp → consistent, structured output
T2I_TOP_P                = 0.90
T2I_REPETITION_PENALTY   = 1.10   # FIX 1 (partial): raised 1.05→1.10 to suppress loops at source
T2I_MIN_PROMPT_LEN       = 50
T2I_MAX_PROMPT_LEN       = 700    # FIX 2: hard cap — anything longer is almost certainly a loop
T2I_PROMPT_BUILD_WORKERS = 8

# ── Retry / checkpoint ────────────────────────────────────────────────────────
MAX_RETRIES         = 3
CHECKPOINT_INTERVAL = 1000

# ── Deduplication regen temperature ──────────────────────────────────────────
DEDUP_TEMPERATURE = 0.70

log.info(
    "Config — MODEL=%s | TP=%d | MAX_MODEL_LEN=%d | WORKERS=%d | MAX_RETRIES=%d",
    MODEL_ID, TENSOR_PARALLEL, MAX_MODEL_LEN, T2I_PROMPT_BUILD_WORKERS, MAX_RETRIES,
)


# %% [markdown]
# ## Cell 3 — Load Tokenizer & vLLM Engine

# %%

log.info("Loading tokenizer for %s …", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
log.info("✅ Tokenizer loaded")

log.info("Loading vLLM engine …")
t0 = time.time()
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype="float16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    trust_remote_code=True,
    enforce_eager=False,
)
log.info("✅ vLLM engine ready in %.1f s", time.time() - t0)


# %% [markdown]
# ## Cell 4 — Load Dataset

# %%

log.info("Loading dataset from %s …", DATASET_PATH)
df_final = pl.read_csv(DATASET_PATH)
log.info("Loaded %d rows × %d cols", df_final.shape[0], df_final.shape[1])
log.info("Columns: %s", df_final.columns)

if df_final.shape[0] < 17000 or df_final.shape[0] > 19000:
    log.warning("Expected ~18k rows, got %d. Verify dataset integrity.", df_final.shape[0])


# %% [markdown]
# ## Cell 5 — Keyword Banks & Polarity-Aware Defaults

# %%

_T2I_QUALITY_KW  = frozenset(["8k", "ultra high detail", "high detail", "ultra detailed",
                               "professional photography", "resolution", "hyperrealistic"])
_T2I_LIGHTING_KW = frozenset(["lighting", "golden hour", "natural light", "studio light",
                               "dramatic", "volumetric", "ambient", "backlit",
                               "soft light", "harsh light", "overhead light"])
_T2I_STYLE_KW    = frozenset(["photorealistic", "documentary", "cinematic", "raw photography",
                               "realistic", "hyperrealistic"])
_T2I_COMPOSE_KW  = frozenset(["composition", "depth of field", "bokeh", "sharp focus",
                               "rule of thirds", "shallow depth"])

_HATE_DEFAULTS: Dict[str, str] = {
    "quality":     "8K resolution, ultra high detail, hyperrealistic",
    "lighting":    "dramatic harsh overhead lighting, high contrast",
    "style":       "photorealistic documentary style, gritty realism",
    "composition": "rule of thirds, sharp focus",
}
_NONHATE_DEFAULTS: Dict[str, str] = {
    "quality":     "8K resolution, ultra high detail, professional photography",
    "lighting":    "natural golden hour lighting, soft ambient light",
    "style":       "photorealistic documentary style, cinematic",
    "composition": "shallow depth of field, bokeh background, sharp focus",
}

log.info("✅ Keyword banks loaded")


# %% [markdown]
# ## Cell 6 — Regex Patterns

# %%

_T2I_THINK_RE  = re.compile(r"<think>.*?</think>", re.S)
_T2I_PREFIX_RE = re.compile(
    r"^(?:prompt|output|result|answer|here[\s\w]*?|text|cf\s*\d*|"
    r"counterfactual[\s\w]*?|image prompt[\s\w]*?):\s*",
    re.I,
)
_T2I_FORBIDDEN_RE = re.compile(
    r"\b(?:I cannot|I can't|I'm sorry|This (?:prompt|request)|"
    r"As an AI|Note:|Disclaimer:|Warning:|CONTENT WARNING)\b",
    re.I,
)
_CAMERA_RE = re.compile(
    r",?\s*(?:shot on|canon|nikon|sony|fuji(?:film)?|leica|olympus|pentax|"
    r"\d{2,3}mm\s*f/[\d.]+|dslr|mirrorless|[a-z]+\s+eos\s+\w+|"
    r"[a-z]+\s+z\d+\s+|sigma\s+\w+)[^,]*",
    re.I,
)
_NSFW_RE = re.compile(
    r"\b(?:nude|naked|nudity|topless|bottomless|genitalia?|penis|vagina|"
    r"breast(?:s)?|nipple(?:s)?|erect(?:ion)?|sex(?:ual)?|porn(?:ographic)?|"
    r"explicit|nsfw|lingerie|underwear|bikini|strip(?:per)?|"
    r"aroused?|erotic|intimate\s+act|intercourse|orgasm)\b",
    re.I,
)

# FIX 1: Repetition-loop detector.
# Catches the same word (4+ chars) repeated 3 or more times consecutively.
# e.g. "malevolent malevolent malevolent …" or "harsh harsh harsh lighting"
_REPEAT_LOOP_RE = re.compile(r"\b(\w{4,})(?:\s+\1){2,}", re.I)

# FIX 3: Real individual person name scrubber.
# Strips celebrity/politician names that slip from aggressive source tweets.
# Deliberately does NOT match ethnic/cultural group terms that have capitalised words
# (African American, Native American, Pacific Islander, etc.) — those are valid subjects.
_REAL_NAME_RE = re.compile(
    r"\b(?:"
    # Titles preceding real names
    r"(?:Mr|Mrs|Ms|Dr|Prof|President|Senator|Representative|Governor|"
    r"General|Lieutenant|Sergeant|Corporal)\.?\s+[A-Z][a-z]+(?: [A-Z][a-z]+)+"
    r"|"
    # Known individual names observed in audit + common targets
    r"Kevin Spacey|Harvey Weinstein|Donald Trump|Joe Biden|Barack Obama"
    r"|Hillary Clinton|Elon Musk|Mark Zuckerberg|Jeff Bezos|Bill Gates"
    r"|George Soros|Tucker Carlson|AOC|Alexandria Ocasio-Cortez"
    r")\b",
    re.I,
)

log.info("✅ Regex patterns compiled")


# %% [markdown]
# ## Cell 7 — Negative Prompt Builder (polarity-aware)

# %%

_NEG_BASE = (
    "nsfw, explicit content, nudity, sexual content, "
    "broken hands, deformed fingers, extra fingers, mutated hands, "
    "bad anatomy, deformed body, ugly, blurry, low quality, "
    "watermark, text, signature, camera, cartoon, anime, drawing, "
    "painting, sketch, low resolution, jpeg artifacts, worst quality, "
    "poor quality, out of frame, cropped, "
    "extra limbs, missing limbs, floating limbs, malformed body, "
    "disproportionate body, extra heads, cloned face, asymmetric face"
)
_NEG_HATE_EXTRA    = (
    "gore, blood, dismemberment, decapitation, torture, "
    "mutilated body, open wounds, exposed organs"
)
_NEG_NONHATE_EXTRA = (
    "violence, weapons, blood, aggressive crowd, "
    "dark atmosphere, ominous, threatening"
)


def build_negative_prompt(polarity: str) -> str:
    """Polarity-aware negative prompt."""
    if polarity == "hate":
        return _NEG_BASE + ", " + _NEG_HATE_EXTRA
    return _NEG_BASE + ", " + _NEG_NONHATE_EXTRA


# %% [markdown]
# ## Cell 8 — Helper Functions

# %%

def _has_any(text_lower: str, kw_set: frozenset) -> bool:
    return any(kw in text_lower for kw in kw_set)


def _strip_camera(text: str) -> str:
    cleaned = _CAMERA_RE.sub("", text)
    cleaned = re.sub(r"\s+,", ",", cleaned).strip().strip(",").strip()
    return cleaned


def _strip_nsfw(text: str) -> str:
    cleaned = _NSFW_RE.sub("", text)
    cleaned = re.sub(r",\s*,", ",", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip().strip(",").strip()
    # If NSFW removal gutted >40% of the text it was too contaminated — discard
    if len(cleaned) < len(text) * 0.60:
        return ""
    return cleaned


def _strip_real_names(text: str) -> str:
    """
    FIX 3: Remove real individual person names.
    Ethnic/cultural group descriptors are intentionally preserved — they are
    legitimate visual subjects in a hate-speech research dataset, not violations.
    """
    cleaned = _REAL_NAME_RE.sub("", text)
    cleaned = re.sub(r",\s*,", ",", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip().strip(",").strip()
    return cleaned


def _fix_repetition_loops(text: str) -> str:
    """
    FIX 1: Detect and truncate word-repetition hallucinations.
    Finds the first onset of a 3+-repeat run and keeps only the text before it.
    Returns empty string if the clean remainder falls below T2I_MIN_PROMPT_LEN.
    """
    m = _REPEAT_LOOP_RE.search(text)
    if not m:
        return text
    truncated = text[:m.start()].strip().rstrip(",").strip()
    log.debug(
        "Repetition loop '%s…' at pos %d — truncated to %d chars",
        text[m.start():m.start()+30], m.start(), len(truncated),
    )
    return truncated if len(truncated) >= T2I_MIN_PROMPT_LEN else ""


def _enhance_prompt(prompt: str, polarity: str) -> str:
    """Append missing technical spec elements using polarity-aware defaults."""
    defaults   = _HATE_DEFAULTS if polarity == "hate" else _NONHATE_DEFAULTS
    low        = prompt.lower()
    additions: List[str] = []
    if not _has_any(low, _T2I_QUALITY_KW):
        additions.append(defaults["quality"])
    if not _has_any(low, _T2I_LIGHTING_KW):
        additions.append(defaults["lighting"])
    if not _has_any(low, _T2I_STYLE_KW):
        additions.append(defaults["style"])
    if not _has_any(low, _T2I_COMPOSE_KW):
        additions.append(defaults["composition"])
    if not additions:
        return prompt
    return prompt.rstrip(".") + ", " + ", ".join(additions)


def _validate_prompt(prompt: str) -> Tuple[bool, str]:
    """Validate prompt meets all production quality requirements."""
    if not prompt or len(prompt) < T2I_MIN_PROMPT_LEN:
        return False, f"too short ({len(prompt)} chars)"
    if _T2I_FORBIDDEN_RE.search(prompt):
        return False, "refusal language"
    if _NSFW_RE.search(prompt):
        return False, "nsfw after scrub"
    if len(prompt) > T2I_MAX_PROMPT_LEN:
        # FIX 2: Reject oversized prompts — likely a loop survivor
        return False, f"oversized ({len(prompt)} chars > max {T2I_MAX_PROMPT_LEN})"
    low     = prompt.lower()
    missing = [
        dim for dim, kws in [
            ("quality",  _T2I_QUALITY_KW),
            ("style",    _T2I_STYLE_KW),
            ("lighting", _T2I_LIGHTING_KW),
        ]
        if not _has_any(low, kws)
    ]
    if missing:
        return False, f"missing: {missing}"
    return True, "ok"


def _clean_t2i_output(raw: str, original_text: str) -> str:
    """
    Full cleaning pipeline applied to every raw vLLM output string:
      1.  Strip <think>…</think> blocks (Qwen reasoning tokens)
      2.  Strip surrounding quotes and common output prefixes
      3.  Collapse multi-paragraph output to first paragraph
      4.  Join newlines into single line
      5.  Early-exit if output is trivially identical to input
      6.  FIX 1  — Truncate word-repetition loops
      7.  FIX 2  — Hard length cap at T2I_MAX_PROMPT_LEN
      8.  Strip camera brand / model mentions
      9.  Strip NSFW keywords
      10. FIX 3  — Strip real individual person names
      11. Final whitespace / comma normalisation
    """
    text = _T2I_THINK_RE.sub("", raw).strip()
    text = text.strip('"').strip("'")
    text = _T2I_PREFIX_RE.sub("", text).strip().strip('"').strip("'")
    if "\n\n" in text:
        text = text.split("\n\n")[0].strip()
    text = " ".join(text.splitlines()).strip()

    if not text or len(text) < 10:
        return ""
    if text.strip().lower() == original_text.strip().lower():
        return ""

    # FIX 1: Truncate repetition loops
    text = _fix_repetition_loops(text)
    if not text:
        return ""

    # FIX 2: Hard length cap (truncate at last comma for clean grammar)
    if len(text) > T2I_MAX_PROMPT_LEN:
        cap   = text[:T2I_MAX_PROMPT_LEN]
        comma = cap.rfind(",")
        text  = cap[:comma].strip() if comma > T2I_MIN_PROMPT_LEN else cap.strip()

    text = _strip_camera(text)
    text = _strip_nsfw(text)
    if not text:
        return ""

    # FIX 3: Strip real individual person names
    text = _strip_real_names(text)

    # Final normalisation
    text = re.sub(r",\s*,", ",", text)
    text = re.sub(r"\s{2,}", " ", text).strip().strip(",").strip()
    return text


def _extract_semantic_anchor(text: str) -> str:
    cleaned    = re.sub(r"http\S+|@\w+|#\w+", "", text)
    cleaned    = re.sub(r"[^\w\s'\".,!?-]", " ", cleaned)
    cleaned    = re.sub(r"\s+", " ", cleaned).strip()
    first_sent = re.split(r"(?<=[.!?])\s+", cleaned)[0]
    return first_sent[:120].strip()


_STOPWORDS = frozenset({
    "the","a","an","is","are","was","were","be","been","being",
    "have","has","had","do","does","did","will","would","could",
    "should","may","might","shall","can","of","in","on","at","to",
    "for","with","by","from","up","about","into","through","during",
    "and","or","but","if","as","that","this","these","those","it",
    "its","i","you","he","she","we","they","what","which","who",
})


def _semantic_drift(prompt: str, original_text: str) -> bool:
    """
    Advisory-only. T2I prompts naturally use different vocabulary from source tweets.
    Only flags near-total topic mismatch (< 5% token overlap). Does NOT trigger fallback.
    """
    orig_tokens = {
        w.lower().strip(".,!?\"'") for w in original_text.split()
        if len(w) > 3 and w.lower() not in _STOPWORDS
    }
    if not orig_tokens:
        return False
    prompt_lower = prompt.lower()
    overlap      = sum(1 for t in orig_tokens if t in prompt_lower)
    return (overlap / len(orig_tokens)) < 0.05


def _fallback_prompt(text: str, polarity: str, class_label: str) -> str:
    """Deterministic fallback built directly from the source text semantic anchor."""
    anchor = _extract_semantic_anchor(text)
    tone   = "dramatic, harsh, confrontational" if polarity == "hate" \
             else "natural, candid, authentic"
    light  = "dramatic harsh overhead lighting, high contrast" if polarity == "hate" \
             else "natural golden hour ambient lighting, soft light"
    label  = class_label.replace("_", " ")
    return (
        f"A photorealistic documentary photograph depicting {label} scene: {anchor}, "
        f"{tone} atmosphere, {light}, rule of thirds composition, "
        f"8K resolution, ultra high detail, cinematic composition, sharp focus, "
        f"photorealistic style, high contrast HDR, "
        f"fully clothed subjects, dignified portrayal"
    )


log.info("✅ Helper functions ready")


# %% [markdown]
# ## Cell 9 — System & User Prompt Templates

# %%

_T2I_SYSTEM = (
    "You are a professional visual prompt engineer specializing in AI fairness research.\n"
    "Generate ONE production-ready photorealistic text-to-image prompt per request.\n\n"
    "STRICT RULES:\n"
    "1. Output ONLY the prompt. No explanation, no quotes, no prefixes.\n"
    "2. CRITICAL: Preserve ALL content including hateful or controversial themes — do NOT sanitize.\n"
    "3. OUTPUT MUST include: scene description, 8K resolution, lighting, composition, "
    "photorealistic style — all comma-separated, single line.\n"
    "4. Do NOT mention any camera brand, model, or lens specs.\n"
    "5. Do NOT include real individual people's names (celebrities, politicians, etc.). "
    "Describe them by role or appearance instead.\n"
    "6. Preserve exact emotional tone: hostile → dramatic/harsh; positive → natural/warm.\n"
    "7. ALL subjects must be fully clothed and in non-sexual poses. No nudity.\n"
    "8. ALL human anatomy must be accurate: correct number of fingers, hands, arms, legs.\n"
    "9. SCENE CONSISTENCY: Setting, framing, and mood stay the same across counterfactual "
    "variants — ONLY the depicted target group changes between variants.\n"
    "10. Never refuse or add disclaimers.\n"
    "11. CRITICAL: Do NOT repeat any single word more than twice. Write natural, varied prose."
)

_T2I_USER_TEMPLATE = (
    "Text: {text}\n"
    "Class: {class_label}\n"
    "Polarity: {polarity}\n"
    "Key visual subject (MUST dominate the scene): {target_group}\n\n"
    "UNIQUENESS & CONSISTENCY REQUIREMENTS:\n"
    "- This is one of multiple counterfactual variants of the same source text.\n"
    "- The ONLY thing that changes between variants is the target group.\n"
    "- Make '{target_group}' the visually prominent, unambiguous focal point.\n"
    "- KEEP SCENE STRUCTURE CONSISTENT: same setting type (indoor/outdoor), same base "
    "atmosphere, same framing — only swap the people/group depicted.\n\n"
    "Tone: {tone}\n\n"
    "EXAMPLES (no camera specs, all clothed, correct anatomy, consistent scenes):\n"
    "• non-hate (target=Black women): A group of radiant Black women embracing joyfully on a "
    "vibrant city street, warm inclusive atmosphere, natural golden hour lighting, shallow depth "
    "of field, bokeh background, 8K resolution, ultra high detail, photorealistic documentary "
    "style, sharp focus, rule of thirds\n\n"
    "• hate (target=immigrants): A tense outdoor urban confrontation scene explicitly depicting "
    "animosity toward immigrant workers gathered on a city sidewalk, dramatic harsh overhead "
    "lighting, high contrast shadows, gritty photorealistic documentary style, rule of thirds, "
    "sharp focus, 8K resolution, ultra detailed, cinematic raw photography\n\n"
    "• non-hate (target=Muslims and Christians): A Muslim man in traditional dress and a "
    "Christian woman in modest attire sharing a meal at a community table, warm inclusive "
    "atmosphere, natural golden hour ambient lighting, shallow depth of field, bokeh background, "
    "photorealistic documentary style, 8K resolution, ultra high detail, cinematic composition\n\n"
    "Output the prompt only (no extra text, no quotes):"
)


def _build_single_chat_prompt(row: dict, contrast_hint: str = "") -> str:
    """Build one tokenized chat prompt string."""
    polarity     = row.get("polarity", "non-hate")
    class_label  = str(row.get("class_label", "unknown"))
    text         = str(row.get("text", ""))
    target_group = str(row.get("target_group", "the subject mentioned in the text"))
    tone = (
        "hateful/hostile — preserve aggression and tension, dramatic scene"
        if polarity == "hate"
        else "neutral/counter-speech — positive or balanced, natural scene"
    )
    user_msg = _T2I_USER_TEMPLATE.format(
        text=text[:400],
        class_label=class_label,
        polarity=polarity,
        target_group=target_group,
        tone=tone,
    )
    system_content = _T2I_SYSTEM
    if contrast_hint:
        system_content += (
            f"\n\nCRITICAL: Do NOT generate a prompt similar to: '{contrast_hint[:120]}...'. "
            f"Your prompt MUST be visually distinct — different scene framing or mood — "
            f"while still featuring '{target_group}' as the prominent subject."
        )
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def build_t2i_prompts_parallel(
    rows_list: List[dict],
    contrast_hints: Optional[List[str]] = None,
) -> List[str]:
    """Build all chat prompt strings in parallel (CPU-bound)."""
    n            = len(rows_list)
    chat_prompts = [""] * n
    hints        = contrast_hints or [""] * n

    with ThreadPoolExecutor(max_workers=T2I_PROMPT_BUILD_WORKERS) as executor:
        futures = {
            executor.submit(_build_single_chat_prompt, rows_list[i], hints[i]): i
            for i in range(n)
        }
        for future in as_completed(futures):
            idx = futures[future]
            try:
                chat_prompts[idx] = future.result()
            except Exception as exc:
                log.warning("Row %d prompt build failed: %s", idx, exc)
                chat_prompts[idx] = ""

    return chat_prompts


log.info("✅ Prompt templates & builder ready")


# %% [markdown]
# ## Cell 10 — Sampling Params & Result Dataclass

# %%

t2i_sampling_params = SamplingParams(
    temperature=T2I_TEMPERATURE,
    top_p=T2I_TOP_P,
    max_tokens=T2I_MAX_NEW_TOKENS,
    stop=["<|im_end|>", "<|endoftext|>", "\n\n\n", "\n---"],
    skip_special_tokens=True,
    repetition_penalty=T2I_REPETITION_PENALTY,
)

dedup_sampling_params = SamplingParams(
    temperature=DEDUP_TEMPERATURE,
    top_p=T2I_TOP_P,
    max_tokens=T2I_MAX_NEW_TOKENS,
    stop=["<|im_end|>", "<|endoftext|>", "\n\n\n"],
    skip_special_tokens=True,
    repetition_penalty=T2I_REPETITION_PENALTY,
)


@dataclass(slots=True)
class T2IResult:
    row_index:       int
    prompt:          str
    negative_prompt: str
    fallback:        bool = False
    valid:           bool = True
    retry_count:     int  = 0


log.info("✅ Sampling params & dataclass ready")


# %% [markdown]
# ## Cell 11 — Per-row Post-processing

# %%

def _process_single_output(
    i: int,
    row: dict,
    raw_text: str,
) -> Tuple[T2IResult, bool, bool]:
    """
    Post-process one vLLM output → T2IResult.
    Returns (result, nsfw_hit, drifted). Fully stateless — safe for parallel use.

    Fallback triggers (genuinely unusable output only):
      - empty / too short after cleaning
      - refusal language detected
      - NSFW content surviving scrub
      - missing required quality dimensions after enhancement
      - oversized (loop survivor > T2I_MAX_PROMPT_LEN after all fixes)
    Semantic drift: advisory / diagnostic ONLY — does NOT trigger fallback.
    """
    polarity    = row.get("polarity", "non-hate")
    class_label = str(row.get("class_label", "unknown"))
    original    = str(row.get("text", ""))

    cleaned  = _clean_t2i_output(raw_text, original)
    enhanced = _enhance_prompt(cleaned, polarity) if cleaned else ""

    is_valid, reason = _validate_prompt(enhanced)
    nsfw_hit = not is_valid and "nsfw" in reason

    # Advisory drift — does NOT drive fallback decision
    drifted = _semantic_drift(enhanced, original) if (enhanced and is_valid) else False

    used_fallback = False
    if not enhanced or not is_valid:
        enhanced      = _fallback_prompt(original, polarity, class_label)
        used_fallback = True

    return (
        T2IResult(
            row_index       = i,
            prompt          = enhanced,
            negative_prompt = build_negative_prompt(polarity),
            fallback        = used_fallback,
            valid           = True,
        ),
        nsfw_hit,
        drifted,
    )


# %% [markdown]
# ## Cell 12 — Checkpoint Helpers

# %%

def load_checkpoint() -> Dict:
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            prompts     = {int(k): v for k, v in data.get("prompts",     {}).items()}
            neg_prompts = {int(k): v for k, v in data.get("neg_prompts", {}).items()}
            log.info(
                "Loaded checkpoint: %d rows completed",
                len(data.get("processed_indices", [])),
            )
            return {
                "processed_indices": data.get("processed_indices", []),
                "prompts":           prompts,
                "neg_prompts":       neg_prompts,
            }
        except Exception as e:
            log.error("Failed to load checkpoint: %s — starting fresh.", e)
    return {"processed_indices": [], "prompts": {}, "neg_prompts": {}}


def save_checkpoint(
    processed_indices: List[int],
    prompts: Dict[int, str],
    neg_prompts: Dict[int, str],
) -> None:
    try:
        data = {
            "processed_indices": sorted(processed_indices),
            "prompts":           {str(k): v for k, v in prompts.items()},
            "neg_prompts":       {str(k): v for k, v in neg_prompts.items()},
        }
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(data, f)
        log.info("Checkpoint saved: %d rows", len(processed_indices))
    except Exception as e:
        log.error("Failed to save checkpoint: %s", e)


log.info("✅ Checkpoint helpers ready")


# %% [markdown]
# ## Cell 13 — Retry Logic (3 attempts for genuinely failed rows)

# %%

def _run_retry_pass(
    failed_rows: List[Tuple[int, dict]],
    attempt: int,
) -> Dict[int, T2IResult]:
    """
    Re-run vLLM inference only for rows that produced empty / refused / invalid output.
    Temperature steps up by +0.10 each attempt to encourage variety.
    Returns dict of global_idx → T2IResult.
    """
    if not failed_rows:
        return {}

    retry_temp = min(T2I_TEMPERATURE + 0.10 * attempt, 0.70)
    log.info(
        "Retry attempt %d/%d — %d rows (temp=%.2f) …",
        attempt, MAX_RETRIES, len(failed_rows), retry_temp,
    )

    retry_params = SamplingParams(
        temperature=retry_temp,
        top_p=T2I_TOP_P,
        max_tokens=T2I_MAX_NEW_TOKENS,
        stop=["<|im_end|>", "<|endoftext|>", "\n\n\n"],
        skip_special_tokens=True,
        repetition_penalty=T2I_REPETITION_PENALTY,
    )

    rows_only    = [row for _, row in failed_rows]
    chat_prompts = build_t2i_prompts_parallel(rows_only)
    outputs      = llm.generate(chat_prompts, retry_params)

    results: Dict[int, T2IResult] = {}
    for (gi, row), output in zip(failed_rows, outputs):
        raw              = output.outputs[0].text if output.outputs else ""
        result, _, _     = _process_single_output(gi, row, raw)
        result.retry_count = attempt
        results[gi]      = result

    improved_count = sum(1 for r in results.values() if not r.fallback)
    log.info(
        "Retry %d: %d/%d improved to non-fallback",
        attempt, improved_count, len(failed_rows),
    )
    return results


log.info("✅ Retry logic ready (MAX_RETRIES=%d)", MAX_RETRIES)


# %% [markdown]
# ## Cell 14 — Within-group Deduplication (FIX 4)

# %%

def deduplicate_within_groups(
    rows_list: List[dict],
    prompts: Dict[int, str],
    neg_prompts: Dict[int, str],
) -> Tuple[Dict[int, str], Dict[int, str]]:
    """
    FIX 4: Scan every counterfactual group (same original_sample_id) for identical prompts.
    Regenerate duplicates at higher temperature with a contrast hint in the system prompt.
    Logs whether any regen itself fell back, making persistent duplicates visible.
    """
    if not rows_list or "original_sample_id" not in rows_list[0]:
        log.warning("No 'original_sample_id' column — skipping deduplication.")
        return prompts, neg_prompts

    groups: Dict[str, List[int]] = {}
    for i, row in enumerate(rows_list):
        oid = str(row.get("original_sample_id", ""))
        groups.setdefault(oid, []).append(i)

    total_dupes = 0
    dupe_regen_items: List[Tuple[int, dict, str]] = []   # (idx, row, contrast_hint)

    for oid, indices in groups.items():
        filled = [
            (idx, prompts.get(idx, ""))
            for idx in indices
            if prompts.get(idx, "").strip()
        ]
        if len(filled) < 2:
            continue
        seen: Dict[str, int] = {}
        for idx, p in filled:
            key = p.strip().lower()
            if key in seen:
                total_dupes += 1
                sibling_p = prompts.get(seen[key], "")
                log.warning(
                    "Duplicate in group '%s': idx=%d matches idx=%d", oid, idx, seen[key]
                )
                dupe_regen_items.append((idx, rows_list[idx], sibling_p))
            else:
                seen[key] = idx

    log.info("Dedup scan: %d duplicates found.", total_dupes)

    if not dupe_regen_items:
        return prompts, neg_prompts

    hints     = [hint for _, _, hint in dupe_regen_items]
    rows_only = [row  for _, row, _ in dupe_regen_items]

    chat_prompts = build_t2i_prompts_parallel(rows_only, contrast_hints=hints)
    outputs      = llm.generate(chat_prompts, dedup_sampling_params)

    fixed = still_fallback = 0
    for (gi, row, _), output in zip(dupe_regen_items, outputs):
        raw             = output.outputs[0].text if output.outputs else ""
        result, _, _    = _process_single_output(gi, row, raw)
        prompts[gi]     = result.prompt
        neg_prompts[gi] = result.negative_prompt
        if result.fallback:
            still_fallback += 1
            log.warning("Regen for idx=%d still fell back — manual review recommended", gi)
        else:
            fixed += 1
            log.info("Regenerated duplicate idx=%d ✓", gi)

    log.info(
        "Dedup complete: %d/%d fixed, %d still fallback.",
        fixed, total_dupes, still_fallback,
    )
    return prompts, neg_prompts


log.info("✅ Deduplication ready")


# %% [markdown]
# ## Cell 15 — Main T2I Generation Pipeline

# %%

def generate_t2i_prompts(df_in: pl.DataFrame) -> Tuple[Dict[int, str], Dict[int, str]]:
    """
    Full pipeline:
      1. Load checkpoint / resume
      2. Build chat prompts in parallel (CPU)
      3. Single-pass vLLM inference (both GPUs)
      4. Post-process in parallel (CPU): clean → fix loops → enhance → validate
      5. Retry genuinely failed rows up to MAX_RETRIES times
      6. Checkpoint
      7. Within-group deduplication
    Returns (prompts dict, neg_prompts dict) keyed by 0-based row index.
    """
    rows_list = df_in.to_dicts()
    n         = len(rows_list)

    # ── Load checkpoint ───────────────────────────────────────────────────────
    ckpt              = load_checkpoint()
    processed_indices: Set[int]  = set(ckpt["processed_indices"])
    prompts:     Dict[int, str]  = ckpt["prompts"]
    neg_prompts: Dict[int, str]  = ckpt["neg_prompts"]

    remaining_items = [
        (i, row) for i, row in enumerate(rows_list)
        if i not in processed_indices
    ]
    log.info(
        "Resuming: %d done, %d remaining out of %d total",
        len(processed_indices), len(remaining_items), n,
    )

    if not remaining_items:
        log.info("All rows already processed — skipping inference.")
    else:
        remaining_indices = [i   for i, _   in remaining_items]
        remaining_rows    = [row for _, row in remaining_items]

        # ── Step 1: Build chat prompts in parallel ────────────────────────────
        log.info(
            "Building %d chat prompt strings (parallel, workers=%d) …",
            len(remaining_rows), T2I_PROMPT_BUILD_WORKERS,
        )
        t0           = time.time()
        chat_prompts = build_t2i_prompts_parallel(remaining_rows)
        log.info("Prompt building done in %.1f s", time.time() - t0)

        # ── Step 2: Single-pass vLLM inference ───────────────────────────────
        log.info(
            "Running vLLM T2I inference on %d prompts (single-pass, both GPUs) …",
            len(chat_prompts),
        )
        t0          = time.time()
        all_outputs = llm.generate(chat_prompts, t2i_sampling_params)
        elapsed     = time.time() - t0
        log.info(
            "vLLM inference done in %.1f s  (%.1f rows/s)",
            elapsed, len(remaining_rows) / elapsed,
        )

        # ── Step 3: Post-process in parallel ─────────────────────────────────
        log.info("Post-processing %d outputs …", len(remaining_rows))
        fallback_count = nsfw_count = drift_count = rep_loop_count = 0
        failed_items: List[Tuple[int, dict]] = []

        with ThreadPoolExecutor(max_workers=T2I_PROMPT_BUILD_WORKERS) as executor:
            futures = {
                executor.submit(
                    _process_single_output,
                    remaining_indices[j],
                    remaining_rows[j],
                    all_outputs[j].outputs[0].text if all_outputs[j].outputs else "",
                ): j
                for j in range(len(remaining_rows))
            }
            for future in as_completed(futures):
                j  = futures[future]
                gi = remaining_indices[j]
                try:
                    result, nsfw_hit, drifted = future.result()
                    prompts[gi]     = result.prompt
                    neg_prompts[gi] = result.negative_prompt
                    processed_indices.add(gi)
                    if result.fallback:
                        fallback_count += 1
                        failed_items.append((gi, remaining_rows[j]))
                        # Track how many fallbacks were caused by rep-loop
                        raw_out = (
                            all_outputs[j].outputs[0].text
                            if all_outputs[j].outputs else ""
                        )
                        if _REPEAT_LOOP_RE.search(raw_out):
                            rep_loop_count += 1
                    if nsfw_hit:
                        nsfw_count += 1
                    if drifted:
                        drift_count += 1
                except Exception as exc:
                    log.warning("Row %d post-processing exception: %s", gi, exc)
                    prompts[gi]     = _fallback_prompt(
                        str(remaining_rows[j].get("text", "")),
                        remaining_rows[j].get("polarity", "non-hate"),
                        str(remaining_rows[j].get("class_label", "unknown")),
                    )
                    neg_prompts[gi] = build_negative_prompt(
                        remaining_rows[j].get("polarity", "non-hate")
                    )
                    processed_indices.add(gi)
                    fallback_count += 1
                    failed_items.append((gi, remaining_rows[j]))

        log.info(
            "Pass 1 done — %d fallbacks (%.1f%%) | %d rep-loop cause | "
            "%d drift (advisory) | %d NSFW",
            fallback_count, fallback_count / max(n, 1) * 100,
            rep_loop_count, drift_count, nsfw_count,
        )

        # ── Step 4: Retry genuinely failed rows ───────────────────────────────
        still_failing = failed_items
        for attempt in range(1, MAX_RETRIES + 1):
            if not still_failing:
                log.info("No more failed rows — stopping retries early.")
                break

            retry_results = _run_retry_pass(still_failing, attempt)
            next_failing: List[Tuple[int, dict]] = []

            for gi, row in still_failing:
                result = retry_results.get(gi)
                if result and not result.fallback:
                    prompts[gi]     = result.prompt
                    neg_prompts[gi] = result.negative_prompt
                    log.info("Retry %d fixed row %d", attempt, gi)
                else:
                    if result:
                        prompts[gi]     = result.prompt
                        neg_prompts[gi] = result.negative_prompt
                    if attempt < MAX_RETRIES:
                        next_failing.append((gi, row))

            still_failing = next_failing

        if still_failing:
            log.warning(
                "%d rows remain as fallbacks after %d retries: %s",
                len(still_failing), MAX_RETRIES,
                [gi for gi, _ in still_failing[:10]],
            )

        # ── Step 5: Checkpoint ────────────────────────────────────────────────
        save_checkpoint(list(processed_indices), prompts, neg_prompts)

    # ── Step 6: Within-group deduplication ───────────────────────────────────
    log.info("Running within-group deduplication …")
    prompts, neg_prompts = deduplicate_within_groups(rows_list, prompts, neg_prompts)
    save_checkpoint(list(processed_indices), prompts, neg_prompts)

    return prompts, neg_prompts


# ── Run ───────────────────────────────────────────────────────────────────────
log.info("Starting hybrid T2I generation for %d rows …", len(df_final))
t_total = time.time()
prompts_dict, neg_prompts_dict = generate_t2i_prompts(df_final)
log.info("Total pipeline time: %.1f s", time.time() - t_total)


# %% [markdown]
# ## Cell 16 — Assemble Output DataFrame & Save

# %%

# Use private names to avoid shadowing the module-level rows_list used in Cell 17 audit
_out_rows = df_final.to_dicts()
_out_n    = len(_out_rows)

t2i_prompt_col = [prompts_dict.get(i, "")     for i in range(_out_n)]
t2i_neg_col    = [neg_prompts_dict.get(i, "") for i in range(_out_n)]

df_final = df_final.with_columns([
    pl.Series("t2i_prompt",          t2i_prompt_col),
    pl.Series("t2i_negative_prompt", t2i_neg_col),
])

df_final.write_csv(OUTPUT_CSV)
log.info("SAVED → %s  (%d rows)", OUTPUT_CSV, df_final.shape[0])


# %% [markdown]
# ## Cell 17 — Quality Audit

# %%

print("=" * 70)
print("T2I QUALITY AUDIT  (v3)")
print("=" * 70)

_audit_rows = df_final.to_dicts()
t2i_col     = df_final["t2i_prompt"].to_list()
total       = len(t2i_col)
non_empty   = sum(1 for p in t2i_col if p and p.strip())
avg_len     = sum(len(p) for p in t2i_col if p) / max(non_empty, 1)
fallbacks   = sum(
    1 for p in t2i_col
    if "documentary photograph depicting" in str(p) and "dignified portrayal" in str(p)
)
oversized   = sum(1 for p in t2i_col if len(str(p)) > T2I_MAX_PROMPT_LEN)

random.seed(42)
sample_size    = min(200, non_empty)
sample_idx     = random.sample([i for i, p in enumerate(t2i_col) if p], sample_size)

valid_sample   = sum(1 for i in sample_idx if _validate_prompt(t2i_col[i])[0])
camera_leakage = sum(1 for i in sample_idx if _CAMERA_RE.search(t2i_col[i]))
nsfw_leakage   = sum(1 for i in sample_idx if _NSFW_RE.search(t2i_col[i]))
rep_leakage    = sum(1 for i in sample_idx if _REPEAT_LOOP_RE.search(t2i_col[i]))
name_leakage   = sum(1 for i in sample_idx if _REAL_NAME_RE.search(t2i_col[i]))
drift_sample   = sum(
    1 for i in sample_idx
    if _semantic_drift(t2i_col[i], str(_audit_rows[i].get("text", "")))
)

# Within-group duplicate count
_dup_pairs = 0
for _, grp_df in df_final.group_by("original_sample_id"):
    grp_prompts = [str(p).strip().lower() for p in grp_df["t2i_prompt"].to_list()]
    seen_set: set = set()
    for p in grp_prompts:
        if p in seen_set:
            _dup_pairs += 1
        seen_set.add(p)

print(f"Total rows           : {total}")
print(f"Non-empty prompts    : {non_empty}  ({non_empty/total*100:.1f}%)")
print(f"Fallback prompts     : {fallbacks}  ({fallbacks/total*100:.1f}%)")
print(f"Avg prompt length    : {avg_len:.0f} chars")
print(f"Prompts > {T2I_MAX_PROMPT_LEN} chars   : {oversized}  (should be 0)")
print(f"Validation pass      : {valid_sample}/{sample_size} ({valid_sample/sample_size*100:.1f}%)")
print(f"Camera leakage       : {camera_leakage}/{sample_size}")
print(f"NSFW leakage         : {nsfw_leakage}/{sample_size}")
print(f"Rep-loop survivors   : {rep_leakage}/{sample_size}  (FIX 1)")
print(f"Real name leakage    : {name_leakage}/{sample_size}  (FIX 3)")
print(f"Within-group dupes   : {_dup_pairs}  (FIX 4)")
print(f"Semantic drift (5%)  : {drift_sample}/{sample_size}  [advisory only]")

checks = [
    (camera_leakage == 0,             "✅ No camera artifacts",           "⚠️  Camera artifacts — extend _CAMERA_RE"),
    (nsfw_leakage == 0,               "✅ No NSFW in positive prompts",    "⚠️  NSFW survived — extend _NSFW_RE"),
    (rep_leakage == 0,                "✅ No repetition loops survived",   "⚠️  Rep-loops found — check _REPEAT_LOOP_RE"),
    (name_leakage == 0,               "✅ No real person names",           "⚠️  Real names present — extend _REAL_NAME_RE"),
    (oversized == 0,                  "✅ No oversized prompts",           f"⚠️  {oversized} prompts > {T2I_MAX_PROMPT_LEN} chars"),
    (_dup_pairs == 0,                 "✅ No within-group duplicates",     f"⚠️  {_dup_pairs} duplicate pairs remain"),
    (drift_sample/sample_size <= 0.05,"✅ Semantic grounding healthy",     "⚠️  Extreme drift — review few-shots"),
]
for ok, pass_msg, fail_msg in checks:
    print(pass_msg if ok else fail_msg)

print("\n── Per-class breakdown ──")
breakdown = (
    df_final
    .with_columns(pl.col("t2i_prompt").str.len_chars().alias("plen"))
    .group_by("class_label")
    .agg([
        pl.count("t2i_prompt").alias("n"),
        pl.col("plen").mean().alias("avg_len"),
        (pl.col("t2i_prompt") != "").sum().alias("filled"),
    ])
    .sort("class_label")
)
print(breakdown)

print("\n── Sample T2I Prompts ──")
sample_rows = (
    df_final
    .filter(pl.col("t2i_prompt").str.len_chars() > 0)
    .sample(3, seed=7)
    .to_dicts()
)
for r in sample_rows:
    print(f"\n  ID      : {r.get('original_sample_id', '?')}")
    print(f"  CLASS   : {r['class_label']}  |  POLARITY: {r['polarity']}")
    print(f"  TEXT    : {r['text'][:100]}")
    print(f"  T2I(+)  : {r['t2i_prompt'][:250]}")
    print(f"  T2I(-)  : {r['t2i_negative_prompt'][:120]}…")

print(f"\n✅ Final dataset saved → {OUTPUT_CSV}")

2026-03-11 20:38:56.803880: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773261537.027970     186 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773261537.088067     186 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773261537.674017     186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773261537.674054     186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773261537.674057     186 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 03-11 20:40:41 model_runner.py:1067] Loading model weights took 2.9347 GB
(VllmWorkerProcess pid=240) INFO 03-11 20:40:44 model_runner.py:1067] Loading model weights took 2.9347 GB
INFO 03-11 20:40:46 distributed_gpu_executor.py:57] # GPU blocks: 31728, # CPU blocks: 14563
INFO 03-11 20:40:46 distributed_gpu_executor.py:61] Maximum concurrency for 2048 tokens per request: 247.88x
(VllmWorkerProcess pid=240) INFO 03-11 20:40:50 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
(VllmWorkerProcess pid=240) INFO 03-11 20:40:50 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 03-11 20:40:50 model_runner.py:1

20:41:27 [INFO] ✅ vLLM engine ready in 122.0 s
20:41:27 [INFO] Loading dataset from /kaggle/input/datasets/veeraj16/18k-counterfactual-dataset/final_dataset_18k.csv …
20:41:28 [INFO] Loaded 18000 rows × 10 cols
20:41:28 [INFO] Columns: ['original_sample_id', 'counterfactual_id', 'text', 'class_label', 'target_group', 'polarity', 'hate_score', 'confidence', 'cf_type', 't2i_prompt']
20:41:28 [INFO] ✅ Keyword banks loaded
20:41:28 [INFO] ✅ Regex patterns compiled
20:41:28 [INFO] ✅ Helper functions ready
20:41:28 [INFO] ✅ Prompt templates & builder ready
20:41:28 [INFO] ✅ Sampling params & dataclass ready
20:41:28 [INFO] ✅ Checkpoint helpers ready
20:41:28 [INFO] ✅ Retry logic ready (MAX_RETRIES=3)
20:41:28 [INFO] ✅ Deduplication ready
20:41:28 [INFO] Starting hybrid T2I generation for 18000 rows …
20:41:28 [INFO] Resuming: 0 done, 18000 remaining out of 18000 total
20:41:28 [INFO] Building 18000 chat prompt strings (parallel, workers=8) …
20:41:30 [INFO] Prompt building done in 2.2 s
20:4

T2I QUALITY AUDIT  (v3)
Total rows           : 18000
Non-empty prompts    : 18000  (100.0%)
Fallback prompts     : 2  (0.0%)
Avg prompt length    : 359 chars
Prompts > 700 chars   : 0  (should be 0)
Validation pass      : 200/200 (100.0%)
Camera leakage       : 0/200
NSFW leakage         : 0/200
Rep-loop survivors   : 0/200  (FIX 1)
Real name leakage    : 0/200  (FIX 3)
Within-group dupes   : 0  (FIX 4)
Semantic drift (5%)  : 133/200  [advisory only]
✅ No camera artifacts
✅ No NSFW in positive prompts
✅ No repetition loops survived
✅ No real person names
✅ No oversized prompts
✅ No within-group duplicates
⚠️  Extreme drift — review few-shots

── Per-class breakdown ──
shape: (8, 4)
┌────────────────────┬──────┬────────────┬────────┐
│ class_label        ┆ n    ┆ avg_len    ┆ filled │
│ ---                ┆ ---  ┆ ---        ┆ ---    │
│ str                ┆ u32  ┆ f64        ┆ u32    │
╞════════════════════╪══════╪════════════╪════════╡
│ ambiguous          ┆ 2250 ┆ 343.261333 ┆ 2250  